# EPOWER Energy Intelligence Demo Setup

**A Hands-On Lab for Snowflake Intelligence**

---

## What is Snowflake Intelligence?

**Snowflake Intelligence** is a ready-to-use agentic AI application with an intuitive, conversational interface that helps business users discover and act on deep insights. It lets users interact with their structured and unstructured enterprise data using natural language.

Snowflake Intelligence uses AI-powered **data agents** to:
- **Understand** questions in natural language
- **Perform** analysis across structured and unstructured data
- **Generate** trusted insights with full traceability
- **Take action** on behalf of users

It bridges the gap between valuable enterprise data and the people who need it, empowering users to move beyond stale dashboards and rigid reports - finding answers independently while respecting Snowflake's robust security and governance policies.

### How It Works

```
User Question ──► Cortex Agent API ──► Orchestration (LLM) ──► Tool Selection
                                                                     │
                      ┌──────────────────┬──────────────────┬────────┘
                      ▼                  ▼                  ▼
              ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
              │CORTEX ANALYST│   │CORTEX SEARCH │   │ CUSTOM TOOLS │
              │ (Text-to-SQL)│   │    (RAG)     │   │  (Functions) │
              │              │   │              │   │              │
              │ Semantic     │   │ Documents &  │   │ Web Scraper  │
              │ Views        │   │ Unstructured │   │ Procedures   │
              └──────────────┘   └──────────────┘   └──────────────┘
                      │                  │                  │
                      └──────────────────┴──────────────────┘
                                         │
                                         ▼
                            Reflection & Response Generation
```

---

## What You'll Build

This notebook guides you through building a complete **Snowflake Intelligence** demo for **EPOWER**, a fictional German Energy Retail (B2C) use case. You will create:

| Component | Purpose |
|-----------|---------|
| **Governed Data Foundation** | Structured tables managed and secured by Snowflake |
| **Semantic Views** | Business context layer enabling natural language queries |
| **Cortex Search Services** | RAG over documents and service tickets |
| **Intelligence Agent** | Autonomous AI agent orchestrating all capabilities |

### How to Use This Notebook

You can either:
- **Run all cells** sequentially for a quick setup (~10 minutes)
- **Run cell-by-cell** to learn about each concept as you go

### Prerequisites

- ACCOUNTADMIN role access (or equivalent privileges)
- A Snowflake account with Cortex features enabled

## Session Setup

Before running any cells, we need to establish a connection to Snowflake. In a Snowflake Workspace, use `get_active_session()` to get the current session context.

In [3]:
# Import python packages
import pandas as pd

# Snowpark
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

# Cortex Functions
import snowflake.cortex as cortex

# Get the active session
session = get_active_session()

## Table of Contents

1. Introduction to EPOWER Demo
2. Role & Compute Cluster (Virtual Warehouse) Setup
3. Database & Schema Creation
4. Internal Stage & Data Upload
5. Data Model - Dimension Tables
6. Data Model - Fact Tables
7. Data Loading from Internal Stage
8. Semantic Views for Cortex Analyst
9. Unstructured Data Processing
10. Cortex Search Services
11. Snowflake Intelligence Agent
12. Verification & Next Steps

---

## 2. Role & Compute Cluster (Virtual Warehouse) Setup

A **compute cluster** (also known as virtual warehouse) provides the computational resources needed to execute queries and load data. We create a dedicated role and compute cluster for this demo.

In [ ]:
%%sql -r warehouse_result
-- Create a dedicated role for the demo (must be done by ACCOUNTADMIN)
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS AI_ENGINEER;

-- Grant role to SYSADMIN for administration
GRANT ROLE AI_ENGINEER TO ROLE SYSADMIN;

-- Grant necessary privileges to the role
GRANT CREATE DATABASE ON ACCOUNT TO ROLE AI_ENGINEER;
GRANT CREATE WAREHOUSE ON ACCOUNT TO ROLE AI_ENGINEER; -- CREATE WAREHOUSE = create compute cluster

In [ ]:
current_user = session.get_current_user()

session.sql(f"GRANT ROLE AI_ENGINEER TO USER {current_user}").collect()
session.sql("USE ROLE AI_ENGINEER").collect()

print(f"✓ Granted role to user {current_user} and switched to AI_ENGINEER")

---

## 3. Database & Schema Creation

### Concept: Database Organization

Snowflake uses a three-tier namespace: `DATABASE.SCHEMA.OBJECT`

```
EPOWER_RETAIL (Database)
    └── DEMO_ASSETS (Schema)
            ├── Tables
            ├── Semantic Views
            ├── Cortex Search Services
            └── Functions/Procedures
```

In [ ]:
%%sql -r db_setup_result
-- Switch into the Role AI_ENGINEER
USE ROLE AI_ENGINEER;

-- Create database and schema
CREATE OR REPLACE DATABASE EPOWER_RETAIL;
USE DATABASE EPOWER_RETAIL;

CREATE SCHEMA IF NOT EXISTS DEMO_ASSETS;
USE SCHEMA DEMO_ASSETS;

CREATE WAREHOUSE IF NOT EXISTS EPOWER_WH
  WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND = 300
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 2
  SCALING_POLICY = 'STANDARD';

-- Set EPOWER_WH as default warehouse for current user
SET MY_USER = CURRENT_USER();
ALTER USER IDENTIFIER($MY_USER) SET DEFAULT_WAREHOUSE = EPOWER_WH;

### File Format Definition

A **File Format** defines how Snowflake should parse incoming data files. This is essential for CSV ingestion.

In [ ]:
%%sql -r file_format_result
-- Create file format for CSV files
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
    ESCAPE = 'NONE'
    ESCAPE_UNENCLOSED_FIELD = '\\'
    DATE_FORMAT = 'YYYY-MM-DD'
    TIMESTAMP_FORMAT = 'YYYY-MM-DD HH24:MI:SS'
    NULL_IF = ('NULL', 'null', '', 'N/A', 'n/a');

---

## 4. Internal Stage & Data Upload

### Why Do We Need an Internal Stage?

In a Snowflake Workspace, files from your repository are accessible to **Python code** via local paths (e.g., `pd.read_csv("data/file.csv")`). However, **SQL commands** like `COPY INTO` and Cortex functions like `PARSE_DOCUMENT` can only read files from **Snowflake stages** - they cannot access workspace local files directly.

Therefore, we create a single internal stage with two subfolders:

```
@ENERGY_STAGE/
    ├── demo_data/           <- Structured data (CSV files)
    │   ├── customer_dim.csv
    │   ├── product_dim.csv
    │   ├── sales_fact.csv
    │   └── ...
    └── unstructured_docs/   <- Unstructured data (PDFs, Markdown)
        ├── energy/
        ├── products/
        └── service/
```

**This approach enables:**
- `COPY INTO` to load CSV data into tables
- `PARSE_DOCUMENT` to extract text from PDFs
- `CORTEX SEARCH` to index document content
- Consistent file access for all SQL-based operations

In [ ]:
%%sql -r stage_result
-- Create internal stage with directory tables enabled
CREATE OR REPLACE STAGE ENERGY_STAGE
    FILE_FORMAT = CSV_FORMAT
    COMMENT = 'Internal stage for structured (CSV) and unstructured (PDF/MD) data'
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
import os
import glob

# Get the workspace root directory (parent of notebooks folder)
workspace_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
print(f"Workspace root: {workspace_root}\n")

# Upload structured data (CSV files)
print("Uploading structured data (CSV files)...")
csv_path = os.path.join(workspace_root, "demo_data", "*.csv")
csv_files = glob.glob(csv_path)
print(f"Found {len(csv_files)} CSV files")
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    session.file.put(file_path, "@ENERGY_STAGE/demo_data/", auto_compress=False, overwrite=True)
    print(f"  ✓ Uploaded: {file_name}")

# Upload unstructured documents (PDF/MD files)
print("\nUploading unstructured documents (PDF/MD files)...")
docs_path = os.path.join(workspace_root, "unstructured_docs", "**", "*")
doc_files = [f for f in glob.glob(docs_path, recursive=True) if f.endswith(('.pdf', '.md'))]
print(f"Found {len(doc_files)} document files")
for file_path in doc_files:
    file_name = os.path.basename(file_path)
    session.file.put(file_path, "@ENERGY_STAGE/unstructured_docs/", auto_compress=False, overwrite=True)
    print(f"  ✓ Uploaded: {file_name}")

session.sql("ALTER STAGE ENERGY_STAGE REFRESH").collect()
print(f"\n✓ Stage refreshed - {len(csv_files)} CSV files and {len(doc_files)} documents uploaded")

---

## 5. Data Model - Dimension Tables

### Concept: Star Schema Design

We use a **Star Schema** - a classic data warehouse design pattern:

- **Dimension Tables**: Descriptive attributes (WHO, WHAT, WHERE, WHEN)
- **Fact Tables**: Measurable events/transactions (HOW MUCH, HOW MANY)

**Benefits:**
- Simple, intuitive queries
- Optimized for analytical workloads
- Perfect for Semantic Views and Cortex Analyst

Our EPOWER demo has **13 dimension tables**:

| Dimension Table | Business Domain |
|-----------------|----------------|
| `product_category_dim` | Energy product categories |
| `product_dim` | Products and tariffs |
| `customer_dim` | Residential and business customers |
| `vendor_dim` | Installation and service partners |
| `account_dim` | Financial accounts |
| `department_dim` | Organizational departments |
| `region_dim` | German geographic regions |
| `sales_rep_dim` | Energy consultants |
| `campaign_dim` | Marketing campaigns |
| `channel_dim` | Sales channels |
| `employee_dim` | Employees |
| `job_dim` | Job positions |
| `location_dim` | Office locations |

In [ ]:
%%sql -r dim_tables_result
-- ==========================================
-- DIMENSION TABLES
-- ==========================================

-- Product Category Dimension
CREATE OR REPLACE TABLE product_category_dim (
    category_key INT PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL,
    vertical VARCHAR(50) NOT NULL
) COMMENT = 'Energy product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility';

-- Product Dimension (Energy products/tariffs)
CREATE OR REPLACE TABLE product_dim (
    product_key INT PRIMARY KEY,
    product_name VARCHAR(200) NOT NULL,
    category_key INT NOT NULL,
    category_name VARCHAR(100),
    vertical VARCHAR(50)
) COMMENT = 'Energy products: EPOWER Strom, Gas, Solar, Wärmepumpe offerings';

-- Customer Dimension (German residential and business customers)
CREATE OR REPLACE TABLE customer_dim (
    customer_key INT PRIMARY KEY,
    customer_name VARCHAR(200) NOT NULL,
    customer_type VARCHAR(50),
    housing_type VARCHAR(100),
    vertical VARCHAR(50),
    address VARCHAR(200),
    city VARCHAR(100),
    state VARCHAR(50),
    zip VARCHAR(20),
    region_key INT
) COMMENT = 'German energy customers with housing type for consumption analysis';

-- Vendor Dimension (Installation and service partners)
CREATE OR REPLACE TABLE vendor_dim (
    vendor_key INT PRIMARY KEY,
    vendor_name VARCHAR(200) NOT NULL,
    vendor_type VARCHAR(100),
    vertical VARCHAR(50),
    address VARCHAR(200),
    city VARCHAR(100),
    state VARCHAR(50),
    zip VARCHAR(20)
) COMMENT = 'Installation partners, service providers, and suppliers';

-- Account Dimension
CREATE OR REPLACE TABLE account_dim (
    account_key INT PRIMARY KEY,
    account_name VARCHAR(100) NOT NULL,
    account_type VARCHAR(50)
);

-- Department Dimension
CREATE OR REPLACE TABLE department_dim (
    department_key INT PRIMARY KEY,
    department_name VARCHAR(100) NOT NULL
) COMMENT = 'EPOWER organizational departments';

-- Region Dimension
CREATE OR REPLACE TABLE region_dim (
    region_key INT PRIMARY KEY,
    region_name VARCHAR(100) NOT NULL
) COMMENT = 'German geographic regions: North, South, West, East';

-- Sales Rep Dimension
CREATE OR REPLACE TABLE sales_rep_dim (
    sales_rep_key INT PRIMARY KEY,
    rep_name VARCHAR(200) NOT NULL,
    hire_date DATE
) COMMENT = 'Energy consultants and sales representatives';

-- Campaign Dimension
CREATE OR REPLACE TABLE campaign_dim (
    campaign_key INT PRIMARY KEY,
    campaign_name VARCHAR(300) NOT NULL,
    objective VARCHAR(100)
) COMMENT = 'Marketing campaigns for energy products';

-- Channel Dimension
CREATE OR REPLACE TABLE channel_dim (
    channel_key INT PRIMARY KEY,
    channel_name VARCHAR(100) NOT NULL
);

-- Employee Dimension
CREATE OR REPLACE TABLE employee_dim (
    employee_key INT PRIMARY KEY,
    employee_name VARCHAR(200) NOT NULL,
    gender VARCHAR(1),
    hire_date DATE
);

-- Job Dimension
CREATE OR REPLACE TABLE job_dim (
    job_key INT PRIMARY KEY,
    job_title VARCHAR(100) NOT NULL,
    job_level INT
);

-- Location Dimension
CREATE OR REPLACE TABLE location_dim (
    location_key INT PRIMARY KEY,
    location_name VARCHAR(200) NOT NULL
);

---

## 6. Data Model - Fact Tables

### Concept: Fact Tables

Fact tables contain:
- **Metrics/Measures**: Numeric values (amount, quantity, consumption)
- **Foreign Keys**: Links to dimension tables
- **Timestamps**: When the event occurred

Our EPOWER demo has **6 fact tables** covering:

| Fact Table | Business Domain |
|------------|----------------|
| `sales_fact` | Energy contracts and sales |
| `billing_history` | Consumption and invoicing |
| `service_logs` | Customer service tickets |
| `finance_transactions` | Financial operations |
| `marketing_campaign_fact` | Campaign performance |
| `hr_employee_fact` | Human resources |

In [ ]:
%%sql -r fact_tables_result
-- ==========================================
-- FACT TABLES
-- ==========================================

-- Sales Fact Table (Energy contracts)
CREATE OR REPLACE TABLE sales_fact (
    sale_id INT PRIMARY KEY,
    date DATE NOT NULL,
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    sales_rep_key INT NOT NULL,
    region_key INT NOT NULL,
    vendor_key INT NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    units INT NOT NULL
) COMMENT = 'Energy contracts and product sales. Amount in EUR, Units = kWh for tariffs or count for hardware';

-- Billing History Table (Energy-specific)
CREATE OR REPLACE TABLE billing_history (
    billing_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    billing_date DATE NOT NULL,
    billing_type VARCHAR(50) NOT NULL,
    consumption_kwh INT NOT NULL,
    amount DECIMAL(10,2) NOT NULL,
    payment_status VARCHAR(50)
) COMMENT = 'Monthly energy consumption and billing. billing_type = Electricity or Gas';

-- Service Logs Table (Customer service tickets)
CREATE OR REPLACE TABLE service_logs (
    log_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    log_date DATE NOT NULL,
    topic VARCHAR(100) NOT NULL,
    category VARCHAR(100),
    description VARCHAR(500),
    sentiment VARCHAR(50),
    channel VARCHAR(50),
    priority VARCHAR(50),
    resolution_date DATE,
    agent_key INT
) COMMENT = 'Customer service tickets. Sentiment = Positiv/Neutral/Negativ';

-- Finance Transactions Table
CREATE OR REPLACE TABLE finance_transactions (
    transaction_id INT PRIMARY KEY,
    date DATE NOT NULL,
    account_key INT NOT NULL,
    department_key INT NOT NULL,
    vendor_key INT NOT NULL,
    product_key INT NOT NULL,
    customer_key INT NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    approval_status VARCHAR(20) DEFAULT 'Pending',
    procurement_method VARCHAR(50),
    approver_id INT,
    approval_date DATE,
    purchase_order_number VARCHAR(50),
    contract_reference VARCHAR(100)
);

-- Marketing Campaign Fact Table
CREATE OR REPLACE TABLE marketing_campaign_fact (
    campaign_fact_id INT PRIMARY KEY,
    date DATE NOT NULL,
    campaign_key INT NOT NULL,
    product_key INT NOT NULL,
    channel_key INT NOT NULL,
    region_key INT NOT NULL,
    spend DECIMAL(10,2) NOT NULL,
    leads_generated INT NOT NULL,
    impressions INT NOT NULL
);

-- HR Employee Fact Table
CREATE OR REPLACE TABLE hr_employee_fact (
    hr_fact_id INT PRIMARY KEY,
    date DATE NOT NULL,
    employee_key INT NOT NULL,
    department_key INT NOT NULL,
    job_key INT NOT NULL,
    location_key INT NOT NULL,
    salary DECIMAL(10,2) NOT NULL,
    attrition_flag INT NOT NULL
);

-- Customer Products Table (Product Ownership - enables cross-domain analysis)
CREATE OR REPLACE TABLE customer_products (
    customer_product_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    category_key INT NOT NULL,
    category_name VARCHAR(100) NOT NULL,
    acquisition_date DATE NOT NULL,
    status VARCHAR(20) DEFAULT 'Active'
) COMMENT = 'Tracks which products each customer owns. Enables queries like avg consumption for heat pump customers';

-- ==========================================
-- SALESFORCE CRM TABLES
-- ==========================================

CREATE OR REPLACE TABLE sf_accounts (
    account_id VARCHAR(20) PRIMARY KEY,
    account_name VARCHAR(200) NOT NULL,
    customer_key INT NOT NULL,
    industry VARCHAR(100),
    vertical VARCHAR(50),
    billing_street VARCHAR(200),
    billing_city VARCHAR(100),
    billing_state VARCHAR(50),
    billing_postal_code VARCHAR(20),
    account_type VARCHAR(50),
    annual_revenue DECIMAL(15,2),
    employees INT,
    created_date DATE
);

CREATE OR REPLACE TABLE sf_opportunities (
    opportunity_id VARCHAR(20) PRIMARY KEY,
    sale_id INT,
    account_id VARCHAR(20) NOT NULL,
    opportunity_name VARCHAR(200) NOT NULL,
    stage_name VARCHAR(100) NOT NULL,
    amount DECIMAL(15,2) NOT NULL,
    probability DECIMAL(5,2),
    close_date DATE,
    created_date DATE,
    lead_source VARCHAR(100),
    type VARCHAR(100),
    campaign_id INT
);

CREATE OR REPLACE TABLE sf_contacts (
    contact_id VARCHAR(20) PRIMARY KEY,
    opportunity_id VARCHAR(20) NOT NULL,
    account_id VARCHAR(20) NOT NULL,
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    email VARCHAR(200),
    phone VARCHAR(50),
    title VARCHAR(100),
    department VARCHAR(100),
    lead_source VARCHAR(100),
    campaign_no INT,
    created_date DATE
);

---

## 7. Data Loading from Internal Stage

### Loading Data with COPY INTO

Now that the CSV files are in the internal stage (`@ENERGY_STAGE/demo_data/`), we use SQL `COPY INTO` commands to load them into the tables we created earlier.

**Benefits of this approach:**
- Native SQL - simple and performant
- Uses the pre-defined `CSV_FORMAT` file format
- `ON_ERROR = 'CONTINUE'` handles minor data issues gracefully

In [ ]:
# Load Dimension Tables from internal stage
dim_tables = [
    "product_category_dim", "product_dim", "customer_dim", "vendor_dim",
    "account_dim", "department_dim", "region_dim", "sales_rep_dim",
    "campaign_dim", "channel_dim", "employee_dim", "job_dim", "location_dim"
]

print("Loading Dimension Tables...\n")
for table in dim_tables:
    result = session.sql(f"COPY INTO {table} FROM @ENERGY_STAGE/demo_data/{table}.csv FILE_FORMAT = CSV_FORMAT ON_ERROR = 'CONTINUE'").collect()
    rows_loaded = result[0]['rows_loaded'] if result else 0
    print(f"  ✓ {table}: {rows_loaded} rows loaded")

print("\n✓ All dimension tables loaded")

In [ ]:
# Load Fact Tables from internal stage
fact_tables = [
    "sales_fact", "billing_history", "service_logs",
    "finance_transactions", "marketing_campaign_fact", "hr_employee_fact",
    "customer_products"
]

print("Loading Fact Tables...\n")
for table in fact_tables:
    result = session.sql(f"COPY INTO {table} FROM @ENERGY_STAGE/demo_data/{table}.csv FILE_FORMAT = CSV_FORMAT ON_ERROR = 'CONTINUE'").collect()
    rows_loaded = result[0]['rows_loaded'] if result else 0
    print(f"  ✓ {table}: {rows_loaded} rows loaded")

print("\n✓ All fact tables loaded")

In [ ]:
# Load Salesforce CRM Tables from internal stage
sf_tables = ["sf_accounts", "sf_opportunities", "sf_contacts"]

print("Loading Salesforce CRM Tables...\n")
for table in sf_tables:
    result = session.sql(f"COPY INTO {table} FROM @ENERGY_STAGE/demo_data/{table}.csv FILE_FORMAT = CSV_FORMAT ON_ERROR = 'CONTINUE'").collect()
    rows_loaded = result[0]['rows_loaded'] if result else 0
    print(f"  ✓ {table}: {rows_loaded} rows loaded")

print("\n✓ All Salesforce tables loaded")

In [ ]:
%%sql -r verify_load_result
-- Verify data loading
SELECT 'DIMENSION TABLES' as category, '' as table_name, NULL as row_count
UNION ALL SELECT '', 'product_category_dim', COUNT(*) FROM product_category_dim
UNION ALL SELECT '', 'product_dim', COUNT(*) FROM product_dim
UNION ALL SELECT '', 'customer_dim', COUNT(*) FROM customer_dim
UNION ALL SELECT '', 'vendor_dim', COUNT(*) FROM vendor_dim
UNION ALL SELECT '', 'region_dim', COUNT(*) FROM region_dim
UNION ALL SELECT '', '', NULL
UNION ALL SELECT 'FACT TABLES', '', NULL
UNION ALL SELECT '', 'sales_fact', COUNT(*) FROM sales_fact
UNION ALL SELECT '', 'billing_history', COUNT(*) FROM billing_history
UNION ALL SELECT '', 'service_logs', COUNT(*) FROM service_logs
UNION ALL SELECT '', 'finance_transactions', COUNT(*) FROM finance_transactions
UNION ALL SELECT '', 'marketing_campaign_fact', COUNT(*) FROM marketing_campaign_fact
UNION ALL SELECT '', 'hr_employee_fact', COUNT(*) FROM hr_employee_fact;

---

## 8. Semantic Views for Cortex Analyst

### What are Semantic Views?

**Semantic Views** bridge the gap between technical data models and business understanding. They enable:

- **Natural Language Queries** - Ask questions in plain German or English
- **Business Vocabulary** - Define synonyms (e.g., "Kunde" = "Customer")
- **Pre-defined Metrics** - Common calculations ready to use
- **Relationship Mapping** - Automatic joins between tables

### How Cortex Analyst Uses Semantic Views

```
┌─────────────────────────────────────────────────────────────┐
│  User Question: "What were total sales by region last Q?"  │
└─────────────────────────────────────────────────────────────┘
                              │
                              ▼
               ┌──────────────────────────┐
               │     CORTEX ANALYST       │
               │  ┌────────────────────┐  │
               │  │   Semantic View    │  │
               │  │  (Business Logic)  │  │
               │  └────────────────────┘  │
               └──────────────────────────┘
                              │
                              ▼
               ┌──────────────────────────┐
               │    Generated SQL Query   │
               └──────────────────────────┘
```

### Our 4 Semantic Views

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  ENERGY_SALES_SEMANTIC_VIEW                                                 │
│  Analyzes: Contracts, Products, Customers, Regions                          │
├─────────────────────────────────────────────────────────────────────────────┤
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐              │
│  │  customer_dim   │  │   product_dim   │  │   region_dim    │              │
│  └────────┬────────┘  └────────┬────────┘  └────────┬────────┘              │
│           │                    │                    │                       │
│           └────────────────────┼────────────────────┘                       │
│                                ▼                                            │
│                    ┌───────────────────────┐                                │
│                    │      sales_fact       │                                │
│                    └───────────────────────┘                                │
│           ┌────────────────────┼────────────────────┐                       │
│           ▼                    ▼                    ▼                       │
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐              │
│  │  sales_rep_dim  │  │   vendor_dim    │  │ product_cat_dim │              │
│  └─────────────────┘  └─────────────────┘  └─────────────────┘              │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  BILLING_SEMANTIC_VIEW                                                      │
│  Analyzes: Energy Consumption, Invoices, Payment Status                     │
├─────────────────────────────────────────────────────────────────────────────┤
│           ┌─────────────────┐                                               │
│           │  customer_dim   │                                               │
│           └────────┬────────┘                                               │
│                    │                                                        │
│                    ▼                                                        │
│         ┌─────────────────────┐                                             │
│         │   billing_history   │                                             │
│         └─────────────────────┘                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  SERVICE_SEMANTIC_VIEW                                                      │
│  Analyzes: Service Tickets, Customer Sentiment, Support Channels            │
├─────────────────────────────────────────────────────────────────────────────┤
│           ┌─────────────────┐                                               │
│           │  customer_dim   │                                               │
│           └────────┬────────┘                                               │
│                    │                                                        │
│                    ▼                                                        │
│         ┌─────────────────────┐                                             │
│         │    service_logs     │                                             │
│         └─────────────────────┘                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  HR_SEMANTIC_VIEW                                                           │
│  Analyzes: Employees, Departments, Salaries, Attrition                      │
├─────────────────────────────────────────────────────────────────────────────┤
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐              │
│  │ department_dim  │  │  employee_dim   │  │    job_dim      │              │
│  └────────┬────────┘  └────────┬────────┘  └────────┬────────┘              │
│           │                    │                    │                       │
│           └────────────────────┼────────────────────┘                       │
│                                ▼                                            │
│                    ┌───────────────────────┐                                │
│                    │   hr_employee_fact    │                                │
│                    └───────────────────────┘                                │
│                                ▲                                            │
│                    ┌───────────┴───────────┐                                │
│                    │     location_dim      │                                │
│                    └───────────────────────┘                                │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
%%sql -r energy_semantic_result
-- ENERGY SALES SEMANTIC VIEW (Contracts and Products)
CREATE OR REPLACE SEMANTIC VIEW EPOWER_RETAIL.DEMO_ASSETS.ENERGY_SALES_SEMANTIC_VIEW
    tables (
        CUSTOMERS as EPOWER_RETAIL.DEMO_ASSETS.CUSTOMER_DIM primary key (CUSTOMER_KEY) 
            with synonyms=('customers','residential customers','commercial customers') 
            comment='Energy customers with housing type for consumption analysis',
        PRODUCTS as EPOWER_RETAIL.DEMO_ASSETS.PRODUCT_DIM primary key (PRODUCT_KEY) 
            with synonyms=('products','tariffs','plans') 
            comment='Energy products: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility',
        PRODUCT_CATEGORIES as EPOWER_RETAIL.DEMO_ASSETS.PRODUCT_CATEGORY_DIM primary key (CATEGORY_KEY)
            with synonyms=('categories','product types')
            comment='Product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility',
        REGIONS as EPOWER_RETAIL.DEMO_ASSETS.REGION_DIM primary key (REGION_KEY) 
            with synonyms=('regions','areas','territories') 
            comment='German regions: North, South, West, East',
        CONTRACTS as EPOWER_RETAIL.DEMO_ASSETS.SALES_FACT primary key (SALE_ID) 
            with synonyms=('contracts','sales','orders','deals') 
            comment='Energy contracts and product sales',
        SALES_REPS as EPOWER_RETAIL.DEMO_ASSETS.SALES_REP_DIM primary key (SALES_REP_KEY) 
            with synonyms=('sales reps','consultants','advisors') 
            comment='Energy consultants and sales representatives',
        VENDORS as EPOWER_RETAIL.DEMO_ASSETS.VENDOR_DIM primary key (VENDOR_KEY) 
            with synonyms=('vendors','partners','installers','suppliers') 
            comment='Installation and service partners'
    )
    relationships (
        PRODUCT_TO_CATEGORY as PRODUCTS(CATEGORY_KEY) references PRODUCT_CATEGORIES(CATEGORY_KEY),
        CONTRACTS_TO_CUSTOMERS as CONTRACTS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY),
        CONTRACTS_TO_PRODUCTS as CONTRACTS(PRODUCT_KEY) references PRODUCTS(PRODUCT_KEY),
        CONTRACTS_TO_REGIONS as CONTRACTS(REGION_KEY) references REGIONS(REGION_KEY),
        CONTRACTS_TO_SALES_REPS as CONTRACTS(SALES_REP_KEY) references SALES_REPS(SALES_REP_KEY),
        CONTRACTS_TO_VENDORS as CONTRACTS(VENDOR_KEY) references VENDORS(VENDOR_KEY),
        CUSTOMERS_TO_REGIONS as CUSTOMERS(REGION_KEY) references REGIONS(REGION_KEY)
    )
    facts (
        CONTRACTS.CONTRACT_AMOUNT as AMOUNT comment='Contract value in EUR',
        CONTRACTS.CONTRACT_UNITS as UNITS comment='Units sold (kWh or count)',
        CONTRACTS.CONTRACT_RECORD as 1 comment='Count of contracts'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME with synonyms=('customer name','name') comment='Customer name',
        CUSTOMERS.CUSTOMER_TYPE as CUSTOMER_TYPE with synonyms=('customer type','type','segment') comment='Residential or Commercial',
        CUSTOMERS.HOUSING_TYPE as HOUSING_TYPE with synonyms=('housing type','building type','dwelling') comment='Housing type for consumption analysis',
        CUSTOMERS.CITY as CITY with synonyms=('city','location') comment='Customer city',
        CUSTOMERS.STATE as STATE with synonyms=('state','region') comment='German state',
        PRODUCTS.PRODUCT_KEY as PRODUCT_KEY,
        PRODUCTS.PRODUCT_NAME as PRODUCT_NAME with synonyms=('product name','tariff name','plan name') comment='Product or tariff name',
        PRODUCTS.VERTICAL as VERTICAL with synonyms=('vertical','segment','business line') comment='Product vertical: Electricity, Gas, Solar, etc.',
        PRODUCT_CATEGORIES.CATEGORY_KEY as CATEGORY_KEY,
        PRODUCT_CATEGORIES.CATEGORY_NAME as CATEGORY_NAME with synonyms=('category','product category') comment='Product category',
        REGIONS.REGION_KEY as REGION_KEY,
        REGIONS.REGION_NAME as REGION_NAME with synonyms=('region','area','territory') comment='German region: North, South, West, East',
        CONTRACTS.SALE_ID as SALE_ID with synonyms=('contract number','contract ID','sale ID') comment='Contract identifier',
        CONTRACTS.DATE as DATE with synonyms=('contract date','date','signing date') comment='Contract date',
        SALES_REPS.SALES_REP_KEY as SALES_REP_KEY,
        SALES_REPS.REP_NAME as REP_NAME with synonyms=('sales rep','consultant','advisor') comment='Sales consultant name',
        VENDORS.VENDOR_KEY as VENDOR_KEY,
        VENDORS.VENDOR_NAME as VENDOR_NAME with synonyms=('vendor','partner','installer') comment='Installation partner'
    )
    metrics (
        CONTRACTS.TOTAL_REVENUE as SUM(contracts.contract_amount) comment='Total contract revenue in EUR',
        CONTRACTS.TOTAL_CONTRACTS as COUNT(contracts.contract_record) comment='Total number of contracts',
        CONTRACTS.AVERAGE_CONTRACT_VALUE as AVG(contracts.contract_amount) comment='Average contract value',
        CONTRACTS.TOTAL_UNITS as SUM(contracts.contract_units) comment='Total units sold'
    )
    comment='Semantic view for energy sales analysis - contracts, products, customers';

In [ ]:
%%sql -r billing_semantic_result
-- BILLING AND CONSUMPTION SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW EPOWER_RETAIL.DEMO_ASSETS.BILLING_SEMANTIC_VIEW
    tables (
        CUSTOMERS as EPOWER_RETAIL.DEMO_ASSETS.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('customers','clients')
            comment='Energy customers',
        BILLING as EPOWER_RETAIL.DEMO_ASSETS.BILLING_HISTORY primary key (BILLING_ID)
            with synonyms=('billing','invoices','bills','consumption')
            comment='Monthly energy billing and consumption'
    )
    relationships (
        BILLING_TO_CUSTOMERS as BILLING(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY)
    )
    facts (
        BILLING.CONSUMPTION as CONSUMPTION_KWH comment='Energy consumption in kWh',
        BILLING.BILLING_AMOUNT as AMOUNT comment='Invoice amount in EUR',
        BILLING.BILLING_RECORD as 1 comment='Count of billing records'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME comment='Customer name',
        CUSTOMERS.HOUSING_TYPE as HOUSING_TYPE comment='Housing type',
        CUSTOMERS.CITY as CITY comment='City',
        BILLING.BILLING_ID as BILLING_ID,
        BILLING.BILLING_DATE as BILLING_DATE with synonyms=('billing date','invoice date') comment='Billing date',
        BILLING.BILLING_TYPE as BILLING_TYPE with synonyms=('energy type','type') comment='Electricity or Gas',
        BILLING.PAYMENT_STATUS as PAYMENT_STATUS with synonyms=('payment status','status') comment='Paid, Open, Overdue'
    )
    metrics (
        BILLING.TOTAL_CONSUMPTION as SUM(billing.consumption) comment='Total consumption in kWh',
        BILLING.AVERAGE_CONSUMPTION as AVG(billing.consumption) comment='Average consumption in kWh',
        BILLING.TOTAL_BILLING_AMOUNT as SUM(billing.billing_amount) comment='Total billing amount',
        BILLING.TOTAL_INVOICES as COUNT(billing.billing_record) comment='Total number of invoices'
    )
    comment='Semantic view for billing and consumption analysis';

In [ ]:
%%sql -r service_semantic_result
-- SERVICE TICKETS SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW EPOWER_RETAIL.DEMO_ASSETS.SERVICE_SEMANTIC_VIEW
    tables (
        CUSTOMERS as EPOWER_RETAIL.DEMO_ASSETS.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('customers')
            comment='Energy customers',
        SERVICE_TICKETS as EPOWER_RETAIL.DEMO_ASSETS.SERVICE_LOGS primary key (LOG_ID)
            with synonyms=('tickets','service requests','support tickets','complaints')
            comment='Customer service tickets and support requests'
    )
    relationships (
        SERVICE_TO_CUSTOMERS as SERVICE_TICKETS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY)
    )
    facts (
        SERVICE_TICKETS.TICKET_RECORD as 1 comment='Count of service tickets'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME comment='Customer name',
        CUSTOMERS.CITY as CITY comment='City',
        SERVICE_TICKETS.LOG_ID as LOG_ID,
        SERVICE_TICKETS.LOG_DATE as LOG_DATE with synonyms=('ticket date','date','created date') comment='Ticket creation date',
        SERVICE_TICKETS.TOPIC as TOPIC with synonyms=('topic','subject') comment='Ticket topic: Smart Meter, Billing, Contract, etc.',
        SERVICE_TICKETS.CATEGORY as CATEGORY with synonyms=('category','type') comment='Request, Complaint, Information',
        SERVICE_TICKETS.DESCRIPTION as DESCRIPTION with synonyms=('description','details','content') comment='Ticket description',
        SERVICE_TICKETS.SENTIMENT as SENTIMENT with synonyms=('sentiment','mood','tone') comment='Positive, Neutral, Negative',
        SERVICE_TICKETS.PRIORITY as PRIORITY with synonyms=('priority','urgency') comment='Low, Medium, High, Critical',
        SERVICE_TICKETS.CHANNEL as CHANNEL with synonyms=('channel','contact method') comment='Phone, Email, Chat, Social Media'
    )
    metrics (
        SERVICE_TICKETS.TOTAL_TICKETS as COUNT(service_tickets.ticket_record) comment='Total service tickets',
        SERVICE_TICKETS.TOTAL_NEGATIVE as SUM(CASE WHEN service_tickets.sentiment = 'Negative' THEN 1 ELSE 0 END) comment='Negative sentiment tickets',
        SERVICE_TICKETS.TOTAL_POSITIVE as SUM(CASE WHEN service_tickets.sentiment = 'Positive' THEN 1 ELSE 0 END) comment='Positive sentiment tickets'
    )
    comment='Semantic view for customer service analysis';

In [ ]:
%%sql -r dataframe_1
-- CUSTOMER ENERGY SEMANTIC VIEW (Cross-Domain: Billing + Products)
-- Enables queries like "average consumption for customers with heat pumps in Hamburg"
CREATE OR REPLACE SEMANTIC VIEW EPOWER_RETAIL.DEMO_ASSETS.CUSTOMER_ENERGY_SEMANTIC_VIEW
    tables (
        CUSTOMERS as EPOWER_RETAIL.DEMO_ASSETS.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('customers','residential customers','commercial customers')
            comment='Energy customers with housing type and location',
        BILLING as EPOWER_RETAIL.DEMO_ASSETS.BILLING_HISTORY primary key (BILLING_ID)
            with synonyms=('billing','invoices','consumption')
            comment='Monthly energy billing and consumption data',
        CUSTOMER_PRODUCTS as EPOWER_RETAIL.DEMO_ASSETS.CUSTOMER_PRODUCTS primary key (CUSTOMER_PRODUCT_ID)
            with synonyms=('customer products','owned products','product ownership')
            comment='Products owned by customers - links to product categories',
        PRODUCTS as EPOWER_RETAIL.DEMO_ASSETS.PRODUCT_DIM primary key (PRODUCT_KEY)
            with synonyms=('products','tariffs')
            comment='Energy products: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility',
        PRODUCT_CATEGORIES as EPOWER_RETAIL.DEMO_ASSETS.PRODUCT_CATEGORY_DIM primary key (CATEGORY_KEY)
            with synonyms=('categories','product categories')
            comment='Product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility'
    )
    relationships (
        BILLING_TO_CUSTOMERS as BILLING(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY),
        CUSTOMER_PRODUCTS_TO_CUSTOMERS as CUSTOMER_PRODUCTS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY),
        CUSTOMER_PRODUCTS_TO_PRODUCTS as CUSTOMER_PRODUCTS(PRODUCT_KEY) references PRODUCTS(PRODUCT_KEY),
        PRODUCTS_TO_CATEGORIES as PRODUCTS(CATEGORY_KEY) references PRODUCT_CATEGORIES(CATEGORY_KEY)
    )
    facts (
        BILLING.CONSUMPTION as CONSUMPTION_KWH comment='Energy consumption in kWh',
        BILLING.BILLING_AMOUNT as AMOUNT comment='Invoice amount in EUR',
        BILLING.BILLING_RECORD as 1 comment='Count of billing records',
        CUSTOMER_PRODUCTS.PRODUCT_RECORD as 1 comment='Product ownership record count'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME comment='Customer name',
        CUSTOMERS.CUSTOMER_TYPE as CUSTOMER_TYPE with synonyms=('customer type','segment') comment='Residential or Commercial',
        CUSTOMERS.HOUSING_TYPE as HOUSING_TYPE with synonyms=('housing type','building type','dwelling') comment='House type for consumption analysis',
        CUSTOMERS.CITY as CITY with synonyms=('city','location') comment='City name',
        CUSTOMERS.STATE as STATE with synonyms=('state','region') comment='North, South, West, East',
        BILLING.BILLING_ID as BILLING_ID,
        BILLING.BILLING_DATE as BILLING_DATE with synonyms=('billing date','invoice date') comment='Billing date',
        BILLING.BILLING_TYPE as BILLING_TYPE with synonyms=('energy type','type') comment='Electricity or Gas',
        BILLING.PAYMENT_STATUS as PAYMENT_STATUS with synonyms=('payment status','status') comment='Paid, Open, Overdue',
        CUSTOMER_PRODUCTS.CUSTOMER_PRODUCT_ID as CUSTOMER_PRODUCT_ID comment='Product ownership record',
        CUSTOMER_PRODUCTS.OWNED_CATEGORY as CATEGORY_NAME with synonyms=('product category','owns') comment='Category of product owned',
        CUSTOMER_PRODUCTS.OWNERSHIP_STATUS as STATUS comment='Active or Inactive',
        PRODUCTS.PRODUCT_KEY as PRODUCT_KEY,
        PRODUCTS.PRODUCT_NAME as PRODUCT_NAME with synonyms=('product name','tariff name') comment='Product name',
        PRODUCT_CATEGORIES.CATEGORY_KEY as CATEGORY_KEY,
        PRODUCT_CATEGORIES.PROD_CATEGORY_NAME as CATEGORY_NAME comment='Product category name'
    )
    metrics (
        BILLING.TOTAL_CONSUMPTION as SUM(BILLING.CONSUMPTION) comment='Total consumption in kWh',
        BILLING.AVERAGE_CONSUMPTION as AVG(BILLING.CONSUMPTION) comment='Average consumption in kWh',
        BILLING.TOTAL_BILLING_AMOUNT as SUM(BILLING.BILLING_AMOUNT) comment='Total billing amount in EUR',
        BILLING.TOTAL_INVOICES as COUNT(BILLING.BILLING_RECORD) comment='Total number of invoices',
        CUSTOMER_PRODUCTS.TOTAL_PRODUCT_OWNERS as COUNT(DISTINCT CUSTOMER_PRODUCTS.CUSTOMER_KEY) comment='Number of customers owning products'
    )
    comment='Cross-domain semantic view combining billing consumption with product ownership. Use for queries like average consumption for heat pump customers.';

In [ ]:
%%sql -r hr_semantic_result
-- HR SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW EPOWER_RETAIL.DEMO_ASSETS.HR_SEMANTIC_VIEW
    tables (
        DEPARTMENTS as EPOWER_RETAIL.DEMO_ASSETS.DEPARTMENT_DIM primary key (DEPARTMENT_KEY)
            with synonyms=('departments','teams')
            comment='Company departments',
        EMPLOYEES as EPOWER_RETAIL.DEMO_ASSETS.EMPLOYEE_DIM primary key (EMPLOYEE_KEY)
            with synonyms=('employees','staff','personnel')
            comment='Employees',
        HR_RECORDS as EPOWER_RETAIL.DEMO_ASSETS.HR_EMPLOYEE_FACT primary key (HR_FACT_ID)
            with synonyms=('hr records','hr data')
            comment='HR fact records',
        JOBS as EPOWER_RETAIL.DEMO_ASSETS.JOB_DIM primary key (JOB_KEY)
            with synonyms=('jobs','positions','roles')
            comment='Job positions',
        LOCATIONS as EPOWER_RETAIL.DEMO_ASSETS.LOCATION_DIM primary key (LOCATION_KEY)
            with synonyms=('locations','offices','sites')
            comment='Office locations'
    )
    relationships (
        HR_TO_DEPARTMENTS as HR_RECORDS(DEPARTMENT_KEY) references DEPARTMENTS(DEPARTMENT_KEY),
        HR_TO_EMPLOYEES as HR_RECORDS(EMPLOYEE_KEY) references EMPLOYEES(EMPLOYEE_KEY),
        HR_TO_JOBS as HR_RECORDS(JOB_KEY) references JOBS(JOB_KEY),
        HR_TO_LOCATIONS as HR_RECORDS(LOCATION_KEY) references LOCATIONS(LOCATION_KEY)
    )
    facts (
        HR_RECORDS.ATTRITION as ATTRITION_FLAG comment='1 = employee left, 0 = active',
        HR_RECORDS.EMPLOYEE_SALARY as SALARY comment='Salary in EUR',
        HR_RECORDS.HR_RECORD as 1 comment='HR record count'
    )
    dimensions (
        DEPARTMENTS.DEPARTMENT_KEY as DEPARTMENT_KEY,
        DEPARTMENTS.DEPARTMENT_NAME as DEPARTMENT_NAME with synonyms=('department','team') comment='Department name',
        EMPLOYEES.EMPLOYEE_KEY as EMPLOYEE_KEY,
        EMPLOYEES.EMPLOYEE_NAME as EMPLOYEE_NAME with synonyms=('employee name','name') comment='Employee name',
        EMPLOYEES.GENDER as GENDER comment='M or F',
        EMPLOYEES.HIRE_DATE as HIRE_DATE with synonyms=('hire date','start date') comment='Hire date',
        HR_RECORDS.HR_FACT_ID as HR_FACT_ID,
        HR_RECORDS.DATE as DATE with synonyms=('date','record date') comment='Record date',
        JOBS.JOB_KEY as JOB_KEY,
        JOBS.JOB_TITLE as JOB_TITLE with synonyms=('position','role','title') comment='Job title',
        JOBS.JOB_LEVEL as JOB_LEVEL with synonyms=('level','grade') comment='Job level',
        LOCATIONS.LOCATION_KEY as LOCATION_KEY,
        LOCATIONS.LOCATION_NAME as LOCATION_NAME with synonyms=('location','office','site') comment='Office location'
    )
    metrics (
        HR_RECORDS.TOTAL_EMPLOYEES as COUNT(hr_records.hr_record) comment='Total employees',
        HR_RECORDS.TOTAL_ATTRITION as SUM(hr_records.attrition) comment='Total attrition',
        HR_RECORDS.AVERAGE_SALARY as AVG(hr_records.employee_salary) comment='Average salary',
        HR_RECORDS.TOTAL_SALARY as SUM(hr_records.employee_salary) comment='Total salary cost'
    )
    comment='Semantic view for HR analytics';

In [ ]:
%%sql -r show_semantic_result
-- Verify semantic views\n
SHOW SEMANTIC VIEWS;

---

## 9. Unstructured Data Processing

### Concept: Cortex Parse Document

**Cortex Parse** is a Snowflake feature that extracts text and structure from unstructured documents:

- **PDF documents**: Contracts, guides, policies
- **Markdown files**: Documentation, FAQs
- **Layout preservation**: Maintains document structure

**Modes:**
- `OCR`: Optical character recognition for scanned documents
- `LAYOUT`: Preserves document layout and structure (recommended)

This parsed content becomes searchable via **Cortex Search**.

In [ ]:
%%sql -r parse_docs_result
-- Parse PDF and Markdown documents
CREATE OR REPLACE TABLE parsed_content AS 
SELECT 
    relative_path,
    BUILD_STAGE_FILE_URL('@EPOWER_RETAIL.DEMO_ASSETS.ENERGY_STAGE', relative_path) as file_url,
    TO_FILE(BUILD_STAGE_FILE_URL('@EPOWER_RETAIL.DEMO_ASSETS.ENERGY_STAGE', relative_path)) file_object,
    SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
        @EPOWER_RETAIL.DEMO_ASSETS.ENERGY_STAGE,
        relative_path,
        {'mode':'LAYOUT'}
    ):content::string as content
FROM directory(@EPOWER_RETAIL.DEMO_ASSETS.ENERGY_STAGE) 
WHERE relative_path ILIKE 'unstructured_docs/%.pdf' 
   OR relative_path ILIKE 'unstructured_docs/%.md';

In [ ]:
%%sql -r verify_parsed_result
-- Verify parsed documents
SELECT relative_path, LEFT(content, 200) as content_preview 
FROM parsed_content 
LIMIT 100;

---

## 10. Cortex Search Services

### What is Cortex Search?

**Cortex Search** is Snowflake's fully managed, low-latency search engine that powers intelligent search experiences over your data. It's the foundation for building **Retrieval Augmented Generation (RAG)** applications and enterprise search solutions.

### Key Capabilities

- **Hybrid Search** - Combines vector (semantic) and keyword (lexical) search for superior results
- **Automatic Embeddings** - Converts text to vectors using Snowflake Arctic embedding models
- **Semantic Reranking** - AI-powered reranking for optimal relevance
- **Auto-Refresh** - Automatically syncs with source data changes
- **Zero Infrastructure** - No embedding pipelines, vector databases, or tuning required

### How It Works

```
┌────────────────────────────────────────────────────────────────────────┐
│                         CORTEX SEARCH                                  │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│   User Query: "Wie installiere ich eine Wärmepumpe?"                   │
│                              │                                         │
│                              ▼                                         │
│   ┌──────────────────────────────────────────────────┐                │
│   │           HYBRID RETRIEVAL ENGINE                 │                │
│   │  ┌─────────────────┐  ┌─────────────────┐        │                │
│   │  │  Vector Search  │  │ Keyword Search  │        │                │
│   │  │   (Semantic)    │  │   (Lexical)     │        │                │
│   │  └────────┬────────┘  └────────┬────────┘        │                │
│   │           └──────────┬─────────┘                 │                │
│   │                      ▼                           │                │
│   │            ┌─────────────────┐                   │                │
│   │            │ Semantic Rerank │                   │                │
│   │            └────────┬────────┘                   │                │
│   └──────────────────────────────────────────────────┘                │
│                              │                                         │
│                              ▼                                         │
│   Ranked Results: [Heat_Pump_Efficiency_Guide.pdf, ...]               │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

### Our Cortex Search Services

We create **4 search services** to power our RAG-enabled AI agent:

| Search Service | Content | Use Case |
|----------------|---------|----------|
| `ENERGY_DOCS_SEARCH` | Energy policies, green power terms | Policy & compliance questions |
| `PRODUCT_DOCS_SEARCH` | Product guides, installation manuals | Technical support |
| `SERVICE_DOCS_SEARCH` | Customer service handbook, FAQs | Support agent assistance |
| `SERVICE_LOGS_SEARCH` | Historical service tickets | Similar issue lookup |

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `ON` | The text column to search |
| `ATTRIBUTES` | Metadata columns to return with results |
| `TARGET_LAG` | How often to refresh the index (e.g., '1 hour') |
| `EMBEDDING_MODEL` | Model for vectorization (default: snowflake-arctic-embed-m-v1.5) |

In [ ]:
%%sql -r dataframe_3
-- First ensure the wrehouse is running as the search service creation references a dynamic table that presumes an active warehouse
ALTER WAREHOUSE EPOWER_WH RESUME;

In [ ]:
-- Search service for energy policy documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_energy_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = EPOWER_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%Energy%'
           OR relative_path ILIKE '%Green_Power%'
           OR relative_path ILIKE '%E_Mobility%'
           OR relative_path ILIKE '%Waermepumpe%'
           OR relative_path ILIKE '%Heat_Pump%'
    );

In [ ]:
-- Search service for product documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_product_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = EPOWER_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%Smart_Meter%'
           OR relative_path ILIKE '%Solar_Battery%'
           OR relative_path ILIKE '%Heat_Pump%'
    );

In [ ]:
-- Search service for customer service documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_service_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = EPOWER_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%Customer_Service%'
           OR relative_path ILIKE '%Invoice%'
           OR relative_path ILIKE '%Vendor_Management%'
    );

In [ ]:
%%sql -r search_logs_result
-- Search service for service logs (structured data search)
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_service_logs
    ON description
    ATTRIBUTES log_id, customer_key, topic, category, sentiment, priority
    WAREHOUSE = EPOWER_WH
    TARGET_LAG = '1 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            log_id,
            customer_key,
            topic,
            category,
            description,
            sentiment,
            priority
        FROM service_logs
    );

In [ ]:
%%sql -r show_search_result
-- Verify search services
SHOW CORTEX SEARCH SERVICES;

### Search Services for High Cardinality Columns

High cardinality columns (like customer names, cities, product names) can be challenging for Cortex Analyst to match exactly. By creating dedicated Cortex Search services for these columns, we enable **dynamic literal retrieval** - the agent can find the exact value that matches a user's natural language query.

For example, when a user asks *"Show invoices for Max Müller"*, the search service helps find the exact customer name in the database.

| Search Service | Column | Purpose |
|----------------|--------|---------|
| `_CA_CUSTOMER_NAME` | customer_dim.customer_name | Match customer names |
| `_CA_CITY` | customer_dim.city | Match German cities |
| `_CA_PRODUCT_NAME` | product_dim.product_name | Match product/tariff names |
| `_CA_VENDOR_NAME` | vendor_dim.vendor_name | Match partner names |
| `_CA_REP_NAME` | sales_rep_dim.rep_name | Match consultant names |

In [ ]:
%%sql -r dataframe_4

CREATE OR REPLACE CORTEX SEARCH SERVICE _CA_CUSTOMER_NAME
  ON CUSTOMER_NAME
  WAREHOUSE = EPOWER_WH
  TARGET_LAG = '14 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT DISTINCT CUSTOMER_NAME
  FROM customer_dim
);

In [ ]:
%%sql -r dataframe_5

CREATE OR REPLACE CORTEX SEARCH SERVICE _CA_CITY
  ON CITY
  WAREHOUSE = EPOWER_WH
  TARGET_LAG = '14 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT DISTINCT CITY
  FROM customer_dim
);

In [ ]:
%%sql -r dataframe_6

CREATE OR REPLACE CORTEX SEARCH SERVICE _CA_PRODUCT_NAME
  ON PRODUCT_NAME
  WAREHOUSE = EPOWER_WH
  TARGET_LAG = '14 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT DISTINCT PRODUCT_NAME
  FROM product_dim
);

In [ ]:
%%sql -r dataframe_7

CREATE OR REPLACE CORTEX SEARCH SERVICE _CA_VENDOR_NAME
  ON VENDOR_NAME
  WAREHOUSE = EPOWER_WH
  TARGET_LAG = '14 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT DISTINCT VENDOR_NAME
  FROM vendor_dim
);

In [ ]:
%%sql -r dataframe_8

CREATE OR REPLACE CORTEX SEARCH SERVICE _CA_REP_NAME
  ON REP_NAME
  WAREHOUSE = EPOWER_WH
  TARGET_LAG = '14 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
  SELECT DISTINCT REP_NAME
  FROM sales_rep_dim
);

---

## 11. Snowflake Intelligence Agent

### What is a Cortex Agent?

**Cortex Agents** are AI orchestrators that plan tasks, use tools to execute them, and generate natural language responses. They combine the power of Large Language Models (LLMs) with your enterprise data to deliver intelligent, context-aware insights.

### Key Capabilities

- **Multi-Tool Orchestration** - Seamlessly switches between structured data (Cortex Analyst) and unstructured data (Cortex Search)
- **Intelligent Planning** - Breaks complex questions into subtasks and routes to the right tools
- **Reflection & Refinement** - Evaluates results after each tool use to improve accuracy
- **Custom Tools** - Extend with stored procedures and UDFs for custom business logic
- **Governed Access** - Respects Snowflake's RBAC, row-access policies, and column-level security

### How Cortex Agents Work

```
┌────────────────────────────────────────────────────────────────────────────┐
│                           CORTEX AGENT WORKFLOW                            │
├────────────────────────────────────────────────────────────────────────────┤
│                                                                            │
│  1. PLANNING                                                               │
│     User Question ──► LLM Orchestrator ──► Task Decomposition              │
│                                                                            │
│  2. TOOL SELECTION & EXECUTION                                             │
│     ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐       │
│     │  CORTEX ANALYST  │  │  CORTEX SEARCH   │  │   CUSTOM TOOLS   │       │
│     │   (Text-to-SQL)  │  │      (RAG)       │  │ (Functions/SPs)  │       │
│     │                  │  │                  │  │                  │       │
│     │  Semantic Views  │  │  Document Search │  │  e.g. Send Email │       │
│     │  Structured Data │  │ Unstructured Data│  │  Business Logic  │       │
│     └────────┬─────────┘  └────────┬─────────┘  └────────┬─────────┘       │
│              │                     │                     │                 │
│  3. REFLECTION                     │                     │                 │
│     ◄────────┴─────────────────────┴─────────────────────┘                 │
│     Evaluate results ──► Iterate if needed ──► Generate response           │
│                                                                            │
│  4. RESPONSE                                                               │
│     Natural language answer with citations, tables, or visualizations      │
│                                                                            │
└────────────────────────────────────────────────────────────────────────────┘
```

### Agent Components

| Component | Description |
|-----------|-------------|
| **Instructions** | Define agent personality, response language, and behavior guidelines |
| **Tools** | Cortex Analyst (semantic views), Cortex Search (documents), Custom (UDFs/SPs) |
| **Tool Resources** | Configuration linking tools to specific data sources |
| **Threads** | Persist conversation context across multiple interactions |
| **Sample Questions** | Seed questions to help users get started |

### Custom Tool: Email Notifications

Enable the agent to send email notifications. This uses Snowflake's built-in `SYSTEM$SEND_EMAIL` functionality.

**Requirements:**
- Email notification integration must exist
- Recipient email must be verified in Snowflake

In [ ]:
%%sql -r email_integration_result
-- Create email notification integration (requires ACCOUNTADMIN)
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE NOTIFICATION INTEGRATION EPOWER_EMAIL_INTEGRATION
    TYPE = EMAIL
    ENABLED = TRUE
    COMMENT = 'Email integration for EPOWER Intelligence Agent';

GRANT USAGE ON INTEGRATION EPOWER_EMAIL_INTEGRATION TO ROLE AI_ENGINEER;

USE ROLE AI_ENGINEER;

In [ ]:
%%sql -r email_sp_result
-- Create stored procedure to send emails (Python)
-- If recipient is NULL or empty, defaults to current user's email
CREATE OR REPLACE PROCEDURE EPOWER_RETAIL.DEMO_ASSETS.SEND_EMAIL_SP(
    RECIPIENT STRING,
    SUBJECT STRING,
    BODY STRING
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'send_email'
EXECUTE AS CALLER
AS
$$
from snowflake.snowpark import Session

def send_email(session: Session, recipient: str, subject: str, body: str) -> str:
    # Default to current user's email if recipient not provided
    if not recipient or recipient.strip() == '':
        result = session.sql("SELECT EMAIL FROM SNOWFLAKE.ACCOUNT_USAGE.USERS WHERE NAME = CURRENT_USER() LIMIT 1").collect()
        if result and result[0]['EMAIL']:
            recipient = result[0]['EMAIL']
        else:
            return 'Error: No recipient provided and current user has no email configured'
    
    session.call(
        'SYSTEM$SEND_EMAIL',
        'EPOWER_EMAIL_INTEGRATION',
        recipient,
        subject,
        body,
        'text/html'
    )
    return f'Email sent to {recipient} with subject: "{subject}"'
$$;

In [ ]:
%%sql -r email_grant_result
-- Grant execute on the email procedure
GRANT USAGE ON PROCEDURE EPOWER_RETAIL.DEMO_ASSETS.SEND_EMAIL_SP(STRING, STRING, STRING) TO ROLE AI_ENGINEER;

### Creating the Intelligence Agent

The agent specification defines:
- **Instructions**: How the agent should behave and respond
- **Tools**: Available capabilities (Cortex Analyst, Cortex Search)
- **Tool Resources**: Configuration for each tool

In [ ]:
%%sql -r create_agent_result
-- Create the Snowflake Intelligence Agent
CREATE OR REPLACE AGENT EPOWER_RETAIL.DEMO_ASSETS.Energy_Chatbot_Agent
WITH PROFILE='{ "display_name": "EPOWER Energy Intelligence Agent" }'
COMMENT=$$ Agent for energy retail analysis: contracts, consumption, service tickets, and documents. $$
FROM SPECIFICATION $$
models:
  orchestration: auto

instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    
    CRITICAL LANGUAGE RULE (HIGHEST PRIORITY):
    Your response language is determined ONLY by the language of the user's question.
    The language of retrieved data, documents, or database content must NOT influence your response language.
    
    - If the user asks in ENGLISH → Respond ENTIRELY in English. Translate any German data/documents.
    - If the user asks in GERMAN → Respond ENTIRELY in German.
    - NEVER mix languages in a single response.
    - When translating German content to English, maintain accuracy of technical terms and numbers.
    
    FORMATTING GUIDELINES:
    - Use tables for structured data with multiple columns
    - Use bullet points for lists of items or key findings
    - Use prose for explanations and summaries
    - Include units (kWh, EUR, %) in all numeric outputs
    - Round numbers appropriately (2 decimals for currency, whole numbers for counts)
    
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, and internal documents.
    
  orchestration: |
    LANGUAGE REMINDER: Before responding, check the user's question language and ensure your ENTIRE response is in that language. Translate retrieved content as needed.
    
    TOOL SELECTION PRIORITY (follow this order):
    
    1. DOCUMENT QUESTIONS (policies, guides, FAQs):
       - Energy policies, terms, subsidies → energy_docs_search
       - Product guides (solar, heat pumps, smart meters) → product_docs_search
       - Service handbook, billing FAQs → service_docs_search
       - Similar past tickets → service_logs_search
    
    2. STRUCTURED DATA QUESTIONS:
       - Consumption + product ownership (e.g., "heat pump customers' usage") → customer_energy_analyst
       - Pure consumption, billing, payments → billing_analyst
       - Contracts, products, sales, regions → energy_sales_analyst
       - Service tickets, complaints, sentiment → service_analyst
       - HR data, salaries, departments → hr_analyst (RESTRICTED - verify authorization)
    
    MULTI-TOOL QUERIES:
    - For questions spanning multiple domains, break into steps
    - Execute tools sequentially, then combine results
    - Example: "Compare solar customer consumption with ticket volume" → customer_energy_analyst + service_analyst
    
    VISUALIZATION RULES:
    - Always use data_to_chart for: trends over time, comparisons, distributions
    - Use charts when result has >5 data points
    - Prefer bar charts for comparisons, line charts for trends, pie charts for composition
    
    DISAMBIGUATION:
    - If query is ambiguous, ask ONE clarifying question before executing
    - If multiple tools could apply, choose the most specific one
    
    EMAIL HANDLING:
    - Default recipient: current Snowflake user's email address
    - Only ask for recipient if user explicitly wants to send to someone else
    - Always confirm email content before sending
    
    ERROR HANDLING:
    - If a tool returns no results, suggest alternative queries or tools
    - If data seems incomplete, mention the limitation in your response
    
  sample_questions:
    - question: "Was ist der durchschnittliche Stromverbrauch fuer Kunden mit Waermepumpen in Hamburg?"
    - question: "Zeige mir alle negativen Service-Tickets zum Thema Smart Meter."
    - question: "Welche Produkte wurden letzten Monat am meisten verkauft?"
    - question: "Vergleiche den Verbrauch zwischen Kunden mit und ohne Waermepumpe."
    - question: "What is the average electricity consumption for customers with heat pumps?"
    - question: "Show me all negative service tickets about smart meters."
    - question: "Which products had the highest sales last month?"
    - question: "Compare consumption between customers with and without heat pumps."
    - question: "What does our Green Power policy say about renewable certificates?"
    - question: "Send me a summary of this month's top issues by email."

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: energy_sales_analyst
      description: |
        Analyzes energy contracts, products, customers and regional performance.
        USE FOR: contract counts, product sales, revenue, regional breakdowns, customer segments.
        EXAMPLES: "How many solar contracts last month?", "Top regions by revenue", "Gas vs electricity sales"
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: billing_analyst
      description: |
        Analyzes energy consumption (kWh), billing and payment status.
        USE FOR: consumption trends, payment rates, billing amounts, usage patterns.
        EXAMPLES: "Average monthly consumption", "Unpaid invoices", "Usage by month"
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: customer_energy_analyst
      description: |
        Analyzes consumption combined with product ownership - the PRIMARY tool for product-specific consumption.
        USE FOR: consumption by product type, comparing customer segments, product impact on usage.
        EXAMPLES: "Heat pump customer consumption", "Solar vs non-solar usage", "Smart home customer patterns"
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: service_analyst
      description: |
        Analyzes customer service tickets, complaints and satisfaction.
        USE FOR: ticket volumes, sentiment analysis, issue categories, resolution times, channel performance.
        EXAMPLES: "Negative tickets this week", "Smart meter complaints", "Average resolution time"
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: hr_analyst
      description: |
        Analyzes employee data, compensation and organizational metrics. RESTRICTED ACCESS.
        USE FOR: headcount, salary analysis, department stats, attrition rates.
        EXAMPLES: "Average salary by department", "Attrition rate", "New hires this quarter"
  - tool_spec:
      type: cortex_search
      name: energy_docs_search
      description: |
        Searches energy policies, Green Power terms and conditions, and subsidy/incentive documents.
        USE FOR: policy questions, legal terms, renewable certificates, government programs.
        NOTE: Documents are in German - translate results to match user's question language.
  - tool_spec:
      type: cortex_search
      name: product_docs_search
      description: |
        Searches technical product documentation and installation guides.
        USE FOR: product specs, installation requirements, compatibility, technical FAQs.
        NOTE: Documents are in German - translate results to match user's question language.
  - tool_spec:
      type: cortex_search
      name: service_docs_search
      description: |
        Searches customer service handbook, standard procedures and billing explanations.
        USE FOR: service procedures, escalation paths, billing questions, FAQ answers.
        NOTE: Documents are in German - translate results to match user's question language.
  - tool_spec:
      type: cortex_search
      name: service_logs_search
      description: |
        Searches historical service tickets for similar past issues and resolutions.
        USE FOR: finding precedents, similar complaints, past solutions, recurring issues.
        NOTE: Content may be in German - translate results to match user's question language.
  - tool_spec:
      type: data_to_chart
      name: data_to_chart
      description: |
        Generates visualizations from query results.
        USE FOR: trends over time, comparisons between categories, distributions, any result with >5 rows.
        CHART TYPES: bar (comparisons), line (trends), pie (composition), scatter (correlations).
  - tool_spec:
      type: generic
      name: send_email
      description: |
        Sends an email with analysis results or summaries.
        DEFAULT RECIPIENT: Current Snowflake user's email (no need to ask).
        Always confirm content before sending. Supports HTML formatting.
      input_schema:
        type: object
        properties:
          recipient:
            type: string
            description: "Recipient email address. Defaults to current user's email if not provided."
          subject:
            type: string
            description: "Email subject line - be descriptive and specific"
          body:
            type: string
            description: "Email content in HTML format. Include data tables, summaries, and key findings."
        required:
          - subject
          - body

tool_resources:
  energy_sales_analyst:
    semantic_view: "EPOWER_RETAIL.DEMO_ASSETS.ENERGY_SALES_SEMANTIC_VIEW"
    execution_environment:
      type: warehouse
      warehouse: "EPOWER_WH"
  billing_analyst:
    semantic_view: "EPOWER_RETAIL.DEMO_ASSETS.BILLING_SEMANTIC_VIEW"
    execution_environment:
      type: warehouse
      warehouse: "EPOWER_WH"
  customer_energy_analyst:
    semantic_view: "EPOWER_RETAIL.DEMO_ASSETS.CUSTOMER_ENERGY_SEMANTIC_VIEW"
    execution_environment:
      type: warehouse
      warehouse: "EPOWER_WH"
  service_analyst:
    semantic_view: "EPOWER_RETAIL.DEMO_ASSETS.SERVICE_SEMANTIC_VIEW"
    execution_environment:
      type: warehouse
      warehouse: "EPOWER_WH"
  hr_analyst:
    semantic_view: "EPOWER_RETAIL.DEMO_ASSETS.HR_SEMANTIC_VIEW"
    execution_environment:
      type: warehouse
      warehouse: "EPOWER_WH"
  energy_docs_search:
    search_service: "EPOWER_RETAIL.DEMO_ASSETS.SEARCH_ENERGY_DOCS"
    max_results: 5
  product_docs_search:
    search_service: "EPOWER_RETAIL.DEMO_ASSETS.SEARCH_PRODUCT_DOCS"
    max_results: 5
  service_docs_search:
    search_service: "EPOWER_RETAIL.DEMO_ASSETS.SEARCH_SERVICE_DOCS"
    max_results: 5
  service_logs_search:
    search_service: "EPOWER_RETAIL.DEMO_ASSETS.SEARCH_SERVICE_LOGS"
    max_results: 5
  send_email:
    type: procedure
    identifier: "EPOWER_RETAIL.DEMO_ASSETS.SEND_EMAIL_SP"
    execution_environment:
      type: warehouse
      warehouse: "EPOWER_WH"
$$;

In [ ]:
%%sql -r dataframe_2
use role accountadmin;


ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_EU';


CREATE SNOWFLAKE INTELLIGENCE if not exists SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT ;

show snowflake intelligences;

GRANT USAGE ON SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT TO ROLE AI_ENGINEER;

GRANT MODIFY ON SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT TO ROLE AI_ENGINEER;


In [ ]:
%%sql -r grant_role_result
-- Assign agent to Snowflake Intelligence and grant access
USE ROLE accountadmin;

ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT EPOWER_RETAIL.DEMO_ASSETS.Energy_Chatbot_Agent;
GRANT USAGE ON AGENT EPOWER_RETAIL.DEMO_ASSETS.Energy_Chatbot_Agent TO ROLE PUBLIC;
GRANT USAGE ON AGENT EPOWER_RETAIL.DEMO_ASSETS.Energy_Chatbot_Agent TO ROLE AI_ENGINEER;

USE ROLE AI_ENGINEER;

---

## 12. Verification & Next Steps

### Setup Complete!

Let's verify everything was created successfully.

In [ ]:
%%sql -r create_agent_result
-- Verification: Summary
SELECT 
    'Energy AI Demo Setup Complete!' as STATUS,
    'EPOWER_RETAIL.DEMO_ASSETS' as DATABASE_SCHEMA,
    'Energy_Chatbot_Agent' as AGENT_NAME;

In [ ]:
%%sql -r grant_agent_result
-- Verification: Tables
SHOW TABLES IN SCHEMA EPOWER_RETAIL.DEMO_ASSETS;

In [ ]:
%%sql -r semantic_views
-- Verification: Semantic Views (5 total)
SHOW SEMANTIC VIEWS IN SCHEMA EPOWER_RETAIL.DEMO_ASSETS;

In [ ]:
%%sql -r search_services
-- Verification: Cortex Search Services (4 main services + internal _CA_ services)
SHOW CORTEX SEARCH SERVICES IN SCHEMA EPOWER_RETAIL.DEMO_ASSETS;

In [ ]:
%%sql -r agents
-- Verification: Agent
SHOW AGENTS IN SCHEMA EPOWER_RETAIL.DEMO_ASSETS;